In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt

from netCDF4 import Dataset as NetCDFFile
import netCDF4 as ncd
import json

In [2]:
from __future__ import print_function
import argparse

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import DataLoader
from model import Net as DBPNLL
from data import get_eval_set
from functools import reduce
from skimage.transform import pyramid_reduce, pyramid_expand

import scipy.io as sio
import time

In [3]:
def normalize_array(data):
    vmax = np.amax(data[np.nonzero(data)])
    vmin = np.amin(data[np.nonzero(data)])
    range_data = vmax - vmin
    
    normalized_data = (data - vmin)/range_data
    threshold = - 1./range_data
    data = np.maximum(normalized_data,threshold)
    data[data == threshold] = -1
    
    return data

In [4]:
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx

In [5]:
# The boundary format is low_lat, left_lon, high_lat and right_lon
def arrays_centered_on_sensor(coordinates, lat, lon, size, xco2_array):
    nb_sensors = len(coordinates[:,1])
    boundaries = np.ones([nb_sensors, 4]).astype(int)
    subdiv_xco2 = {'centered_arrays':[]}
    for i in range(nb_sensors):
        boundaries[i,0] = find_nearest(lat, coordinates[i, 0] - size)
        boundaries[i,1] = find_nearest(lon, coordinates[i, 1] - size)
        boundaries[i,2] = find_nearest(lat, coordinates[i, 0] + size)
        boundaries[i,3] = find_nearest(lon, coordinates[i, 1] + size)
        subdiv_xco2['centered_arrays'].append(xco2_array[boundaries[i, 0]:boundaries[i, 2],
                                                         boundaries[i, 1]:boundaries[i, 3]])
    return subdiv_xco2, boundaries

In [6]:
# The boundary format is low_lat, left_lon, high_lat and right_lon
def export_coordinates(bounds, lat, lon):
    nb_arrays = len(bounds)
    export_coord = {'coordinates':[]}
    for i in range(nb_arrays):
        export_coord['coordinates'].append([lat[bounds[i,0]], lon[bounds[i,1]], lat[bounds[i,2]], lon[bounds[i,3]]])
    return export_coord

## Information
At low resolution:  
Longitude resolution: 0.625º  
Latitude resolution: 0.5º

In [7]:
file = NetCDFFile('/data/andria/OCO-2/oco2_GEOS_L3CO2_day_20150511_B10206Ar.nc4')

In [8]:
xco2 = np.asarray(file.variables['XCO2'][:])
xco2 = np.squeeze(xco2)
lat = np.asarray(file.variables['lat'][:])
lon = np.asarray(file.variables['lon'][:])

In [9]:
lon_reshaped = np.reshape(lon, (1, 576))
lat_reshaped = np.reshape(lat, (361, 1))

In [10]:
xco2_norm = normalize_array(xco2)

In [11]:
coordinates = np.load('coordinates.npy')
centered_on_sensors, boundaries = arrays_centered_on_sensor(coordinates, lat, lon, 10, xco2_norm)

In [12]:
to_export = export_coordinates(boundaries.astype(int), lat, lon)
with open("../../../emissions_data/coordinates.json", "w") as outfile:
    json.dump(to_export, outfile)

In [13]:
for i in range(len(centered_on_sensors['centered_arrays'])):
    np.save('Input/XCO2/centered_{}.npy'.format(i), centered_on_sensors['centered_arrays'][i])

In [14]:
model_w_16 = 'weights/MOD_tensorese-hivemindDBPNLL1channel_16_MSE.pth'

gpus_list=range(1)
cuda = True
if cuda and not torch.cuda.is_available():
    raise Exception("No GPU found, please run without --cuda")

torch.manual_seed(123)
if cuda:
    torch.cuda.manual_seed(123)

print('===> Loading datasets')
test_set = get_eval_set(os.path.join('Input','XCO2'), 16)
testing_data_loader = DataLoader(dataset=test_set, num_workers=1, batch_size=1, shuffle=False)

print('===> Building models')
model_16 = DBPNLL(num_channels=1, base_filter=64,  feat = 256, num_stages=10, scale_factor=16)

if cuda:
    model_16 = torch.nn.DataParallel(model_16, device_ids=gpus_list)

model_16.load_state_dict(torch.load(model_w_16, map_location=lambda storage, loc: storage))
print('Pre-trained SR model is loaded.')

if cuda:
    model_16 = model_16.cuda(gpus_list[0])

def eval():
    model_16.eval()
    for batch in testing_data_loader:
        with torch.no_grad():
            input, name = Variable(batch[0]), batch[1]
        if cuda:
            input = input.cuda(gpus_list[0])

        t0 = time.time()
        with torch.no_grad():
            prediction_16 = model_16(input)
        t1 = time.time()
        name_16 = name[0].replace('.npy', '_16.npy')
        print("===> Processing: %s || Path: 1-16 || Timer: %.4f sec." % (name_16, (t1 - t0)))
        
        save_img(prediction_16.cpu().data, name_16)
        
def save_img(img, img_name):
    save_arr = img.numpy().squeeze(axis = 0).squeeze(axis = 0)
    # save img
    save_dir=os.path.join('Results/','XCO2')
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    save_fn = save_dir +'/'+ img_name
    np.save(save_fn, save_arr)
    
eval()

===> Loading datasets
===> Building models
Pre-trained SR model is loaded.
===> Processing: centered_0_16.npy || Path: 1-16 || Timer: 1.1468 sec.
===> Processing: centered_1_16.npy || Path: 1-16 || Timer: 0.0059 sec.
===> Processing: centered_2_16.npy || Path: 1-16 || Timer: 0.0059 sec.
===> Processing: centered_3_16.npy || Path: 1-16 || Timer: 0.0057 sec.
===> Processing: centered_4_16.npy || Path: 1-16 || Timer: 0.0059 sec.
===> Processing: centered_5_16.npy || Path: 1-16 || Timer: 0.0057 sec.
===> Processing: centered_6_16.npy || Path: 1-16 || Timer: 0.0057 sec.
===> Processing: centered_7_16.npy || Path: 1-16 || Timer: 0.0059 sec.
===> Processing: centered_8_16.npy || Path: 1-16 || Timer: 0.0058 sec.
===> Processing: centered_9_16.npy || Path: 1-16 || Timer: 0.0057 sec.
===> Processing: centered_10_16.npy || Path: 1-16 || Timer: 0.0057 sec.
